In [1]:
# ============================================================
# SURVEY AUDIO NORMALIZATION — oddzielny plik, bez SpeechBrain
# ============================================================

import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
import pandas as pd, re

RAVDESS_DIR    = Path(r"D:\Bachelor Project Code\Audio_Speech_Actors_01-24")
ELEVENLABS_DIR = Path(r"D:\Bachelor Project Code\ElevenLabs_Speech_Voices")
OUTPUT_DIR     = Path(r"D:\Bachelor Project Code\Survey_Audio")
CODE_DIR       = Path(r"D:\Bachelor Project Code\Code")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_RMS = 0.05
np.random.seed(42)

RAVDESS_EMO_MAP = {
    "01":"neutral","03":"happy","04":"sad",
    "05":"angry","06":"fearful","07":"disgust"
}
EMO_TO_CODE = {v:k for k,v in RAVDESS_EMO_MAP.items()}

def normalize_and_save(src, dst, target_rms=TARGET_RMS):
    y, sr = librosa.load(str(src), sr=None, mono=True)
    rms = np.sqrt(np.mean(y**2))
    if rms > 0:
        y = y * (target_rms / rms)
    y = np.clip(y, -1.0, 1.0)
    sf.write(str(dst), y, sr)
    return rms

# ── PLAN PLIKÓW ───────────────────────────────────────────────
# S1: RAVDESS, Actor 11(M)/16(F), normal intensity, kids/dogs naprzemiennie
s1_plan = [
    ("neutral", "11", "01"),
    ("happy",   "16", "02"),
    ("sad",     "11", "01"),
    ("angry",   "16", "02"),
    ("fearful", "11", "01"),
    ("disgust", "16", "02"),
]

# S2: ElevenLabs Ryan(M)/Sarah(F), Natural, kids/dogs naprzemiennie
s2_plan = [
    ("neutral", "ryan01",  "kids"),
    ("happy",   "sarah01", "dogs"),
    ("sad",     "ryan01",  "kids"),
    ("angry",   "sarah01", "dogs"),
    ("fearful", "ryan01",  "kids"),
    ("disgust", "sarah01", "dogs"),
]

# S3: Mix — inne pliki niż S1/S2
s3_ravdess = [
    ("happy",   "11", "02", "02"),
    ("disgust", "16", "01", "02"),
    ("angry",   "11", "02", "02"),
    ("sad",     "16", "01", "02"),
]
s3_el = [
    ("fearful", "sarah01", "dogs", "g2"),
    ("neutral", "ryan01",  "kids", "g2"),
    ("angry",   "ryan01",  "dogs", "g2"),
    ("happy",   "sarah01", "kids", "g2"),
]

# ── BUDUJ LISTĘ PLIKÓW ────────────────────────────────────────
all_files = []

for emotion, actor, stmt in s1_plan:
    code  = EMO_TO_CODE[emotion]
    fname = f"03-01-{code}-01-{stmt}-01-{actor}.wav"
    fpath = RAVDESS_DIR / f"Actor_{actor}" / fname
    all_files.append({
        "section":1, "source":"human", "emotion":emotion,
        "gender":"M" if int(actor)%2==1 else "F",
        "statement":"kids" if stmt=="01" else "dogs",
        "original_path":str(fpath),
        "survey_filename":f"S1_{emotion}_{'M' if int(actor)%2==1 else 'F'}.wav",
        "exists": fpath.exists()
    })

for emotion, voice, stmt in s2_plan:
    candidates = list(ELEVENLABS_DIR.rglob(
        f"ELv3_{voice}_N_{emotion}_{stmt}_g1.wav"))
    fpath = candidates[0] if candidates else None
    all_files.append({
        "section":2, "source":"ai", "emotion":emotion,
        "gender":"M" if "ryan" in voice else "F",
        "statement":stmt,
        "original_path":str(fpath) if fpath else "MISSING",
        "survey_filename":f"S2_{emotion}_{'M' if 'ryan' in voice else 'F'}.wav",
        "exists": fpath is not None
    })

for emotion, actor, stmt, rep in s3_ravdess:
    code  = EMO_TO_CODE[emotion]
    fname = f"03-01-{code}-01-{stmt}-{rep}-{actor}.wav"
    fpath = RAVDESS_DIR / f"Actor_{actor}" / fname
    all_files.append({
        "section":3, "source":"human", "emotion":emotion,
        "gender":"M" if int(actor)%2==1 else "F",
        "statement":"kids" if stmt=="01" else "dogs",
        "original_path":str(fpath),
        "survey_filename":f"S3_H_{emotion}_{'M' if int(actor)%2==1 else 'F'}.wav",
        "exists": fpath.exists()
    })

for emotion, voice, stmt, gen in s3_el:
    candidates = list(ELEVENLABS_DIR.rglob(
        f"ELv3_{voice}_N_{emotion}_{stmt}_{gen}.wav"))
    fpath = candidates[0] if candidates else None
    all_files.append({
        "section":3, "source":"ai", "emotion":emotion,
        "gender":"M" if "ryan" in voice else "F",
        "statement":stmt,
        "original_path":str(fpath) if fpath else "MISSING",
        "survey_filename":f"S3_AI_{emotion}_{'M' if 'ryan' in voice else 'F'}.wav",
        "exists": fpath is not None
    })

# ── SPRAWDŹ PRZED NORMALIZACJĄ ────────────────────────────────
df = pd.DataFrame(all_files)
missing = df[~df["exists"]]
if len(missing) > 0:
    print(f"MISSING FILES ({len(missing)}):")
    print(missing[["section","source","emotion","original_path"]].to_string())
    print("\nFix missing files before continuing.")
else:
    print(f"All {len(df)} files found. Starting normalization...\n")
    
    for _, row in df.iterrows():
        src = Path(row["original_path"])
        dst = OUTPUT_DIR / row["survey_filename"]
        orig_rms = normalize_and_save(src, dst)
        print(f"  ✓ {row['survey_filename']:45s} rms: {orig_rms:.4f} → {TARGET_RMS}")

    df["output_path"] = df["survey_filename"].apply(
        lambda x: str(OUTPUT_DIR / x))
    df.to_csv(CODE_DIR / "survey_files_final.csv", index=False)

    print(f"\n{'='*55}")
    print(f"DONE — {len(df)} files saved to:\n  {OUTPUT_DIR}")
    print(f"\nBalance:")
    print(f"  Source:    {df.groupby('source').size().to_dict()}")
    print(f"  Gender:    {df.groupby('gender').size().to_dict()}")
    print(f"  Statement: {df.groupby('statement').size().to_dict()}")
    print(f"  Section:   {df.groupby('section').size().to_dict()}")

All 20 files found. Starting normalization...



d:\Bachelor Project Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✓ S1_neutral_M.wav                              rms: 0.0030 → 0.05
  ✓ S1_happy_F.wav                                rms: 0.0118 → 0.05
  ✓ S1_sad_M.wav                                  rms: 0.0031 → 0.05
  ✓ S1_angry_F.wav                                rms: 0.0204 → 0.05
  ✓ S1_fearful_M.wav                              rms: 0.0052 → 0.05
  ✓ S1_disgust_F.wav                              rms: 0.0085 → 0.05
  ✓ S2_neutral_M.wav                              rms: 0.0815 → 0.05
  ✓ S2_happy_F.wav                                rms: 0.1048 → 0.05
  ✓ S2_sad_M.wav                                  rms: 0.1065 → 0.05
  ✓ S2_angry_F.wav                                rms: 0.1606 → 0.05
  ✓ S2_fearful_M.wav                              rms: 0.0748 → 0.05
  ✓ S2_disgust_F.wav                              rms: 0.1047 → 0.05
  ✓ S3_H_happy_M.wav                              rms: 0.0073 → 0.05
  ✓ S3_H_disgust_F.wav                            rms: 0.0083 → 0.05
  ✓ S3_H_angry_M.wav              

In [2]:
# ============================================================
# AUDIO PREVIEW — wszystkie głosy per emocja
# Posłuchaj kolejno żeby porównać
# ============================================================

from pathlib import Path
import pandas as pd

SURVEY_DIR = Path(r"D:\Bachelor Project Code\Survey_Audio")
CODE_DIR   = Path(r"D:\Bachelor Project Code\Code")

df = pd.read_csv(CODE_DIR / "survey_files_final.csv")

print("SURVEY AUDIO FILES — posłuchaj kolejno:\n")
print(f"{'Sekcja':<10} {'Źródło':<8} {'Emocja':<10} {'Płeć':<6} "
      f"{'Zdanie':<8} {'Plik'}")
print("-" * 75)

for _, row in df.sort_values(["section","source","emotion"]).iterrows():
    print(f"  S{row['section']:<9} {row['source']:<8} {row['emotion']:<10} "
          f"{row['gender']:<6} {row['statement']:<8} {row['survey_filename']}")

print(f"\nFolder: {SURVEY_DIR}")
print("\nOtwórz folder i przesłuchaj pliki w kolejności sekcji.")

SURVEY AUDIO FILES — posłuchaj kolejno:

Sekcja     Źródło   Emocja     Płeć   Zdanie   Plik
---------------------------------------------------------------------------
  S1         human    angry      F      dogs     S1_angry_F.wav
  S1         human    disgust    F      dogs     S1_disgust_F.wav
  S1         human    fearful    M      kids     S1_fearful_M.wav
  S1         human    happy      F      dogs     S1_happy_F.wav
  S1         human    neutral    M      kids     S1_neutral_M.wav
  S1         human    sad        M      kids     S1_sad_M.wav
  S2         ai       angry      F      dogs     S2_angry_F.wav
  S2         ai       disgust    F      dogs     S2_disgust_F.wav
  S2         ai       fearful    M      kids     S2_fearful_M.wav
  S2         ai       happy      F      dogs     S2_happy_F.wav
  S2         ai       neutral    M      kids     S2_neutral_M.wav
  S2         ai       sad        M      kids     S2_sad_M.wav
  S3         ai       angry      M      dogs     S3_AI_

In [4]:
# ============================================================
# AUDIO PREVIEW — posortowane do odsłuchu per głos/płeć
# Folder: Survey_Preview/
# ============================================================

import shutil
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
import pandas as pd, re

RAVDESS_DIR    = Path(r"D:\Bachelor Project Code\Audio_Speech_Actors_01-24")
ELEVENLABS_DIR = Path(r"D:\Bachelor Project Code\ElevenLabs_Speech_Voices")
PREVIEW_DIR    = Path(r"D:\Bachelor Project Code\Survey_Preview")
PREVIEW_DIR.mkdir(exist_ok=True)

TARGET_RMS = 0.05
EMOTIONS   = ["neutral", "happy", "sad", "angry", "fearful", "disgust"]
EMO_TO_CODE = {"neutral":"01","happy":"03","sad":"04",
               "angry":"05","fearful":"06","disgust":"07"}

def norm_save(src, dst):
    y, sr = librosa.load(str(src), sr=None, mono=True)
    rms = np.sqrt(np.mean(y**2))
    if rms > 0:
        y = y * (TARGET_RMS / rms)
    sf.write(str(dst), np.clip(y,-1,1), sr)

# Kolejność odsłuchu: 01=neutral → 03=happy → 04=sad
#                     → 05=angry → 06=fearful → 07=disgust

groups = {
    "01_Human_M_Actor11": [
        ("neutral","11","01","01"),  # kids, rep1
        ("happy",  "11","01","01"),
        ("sad",    "11","01","01"),
        ("angry",  "11","01","01"),
        ("fearful","11","01","01"),
        ("disgust","11","01","01"),
    ],
    "02_Human_F_Actor16": [
        ("neutral","16","01","01"),
        ("happy",  "16","01","01"),
        ("sad",    "16","01","01"),
        ("angry",  "16","01","01"),
        ("fearful","16","01","01"),
        ("disgust","16","01","01"),
    ],
    "03_AI_M_Ryan_Natural": [
        ("neutral","ryan01","kids","g1"),
        ("happy",  "ryan01","kids","g1"),
        ("sad",    "ryan01","kids","g1"),
        ("angry",  "ryan01","kids","g1"),
        ("fearful","ryan01","kids","g1"),
        ("disgust","ryan01","kids","g1"),
    ],
    "04_AI_F_Sarah_Natural": [
        ("neutral","sarah01","kids","g1"),
        ("happy",  "sarah01","kids","g1"),
        ("sad",    "sarah01","kids","g1"),
        ("angry",  "sarah01","kids","g1"),
        ("fearful","sarah01","kids","g1"),
        ("disgust","sarah01","kids","g1"),
    ],
}

print("Generating preview files...\n")

for group_name, plan in groups.items():
    group_dir = PREVIEW_DIR / group_name
    group_dir.mkdir(exist_ok=True)
    print(f"{group_name}:")

    for i, entry in enumerate(plan):
        emotion = entry[0]
        num = i + 1

        if group_name.startswith("01_") or group_name.startswith("02_"):
            # RAVDESS
            _, actor, stmt, rep = entry
            code  = EMO_TO_CODE[emotion]
            fname = f"03-01-{code}-01-{stmt}-{rep}-{actor}.wav"
            src   = RAVDESS_DIR / f"Actor_{actor}" / fname
        else:
            # ElevenLabs
            _, voice, stmt, gen = entry
            candidates = list(ELEVENLABS_DIR.rglob(
                f"ELv3_{voice}_N_{emotion}_{stmt}_{gen}.wav"))
            src = candidates[0] if candidates else None

        dst_name = f"{num:02d}_{emotion}.wav"
        dst      = group_dir / dst_name

        if src and Path(src).exists():
            norm_save(src, dst)
            print(f"  ✓ {dst_name}")
        else:
            print(f"  ✗ MISSING: {emotion}")

print(f"\nDONE — otwórz folder i słuchaj po kolei:")
print(f"  {PREVIEW_DIR}")
print(f"\nKolejność odsłuchu:")
print(f"  1. 01_Human_M_Actor11   → 6 emocji Actor 11 (M)")
print(f"  2. 02_Human_F_Actor16   → 6 emocji Actor 16 (F)")
print(f"  3. 03_AI_M_Ryan_Natural → 6 emocji Ryan (M)")
print(f"  4. 04_AI_F_Sarah_Natural→ 6 emocji Sarah (F)")
print(f"\nJeśli emocja brzmi zbyt podobnie do neutral →")

Generating preview files...

01_Human_M_Actor11:
  ✓ 01_neutral.wav
  ✓ 02_happy.wav
  ✓ 03_sad.wav
  ✓ 04_angry.wav
  ✓ 05_fearful.wav
  ✓ 06_disgust.wav
02_Human_F_Actor16:
  ✓ 01_neutral.wav
  ✓ 02_happy.wav
  ✓ 03_sad.wav
  ✓ 04_angry.wav
  ✓ 05_fearful.wav
  ✓ 06_disgust.wav
03_AI_M_Ryan_Natural:
  ✓ 01_neutral.wav
  ✓ 02_happy.wav
  ✓ 03_sad.wav
  ✓ 04_angry.wav
  ✓ 05_fearful.wav
  ✓ 06_disgust.wav
04_AI_F_Sarah_Natural:
  ✓ 01_neutral.wav
  ✓ 02_happy.wav
  ✓ 03_sad.wav
  ✓ 04_angry.wav
  ✓ 05_fearful.wav
  ✓ 06_disgust.wav

DONE — otwórz folder i słuchaj po kolei:
  D:\Bachelor Project Code\Survey_Preview

Kolejność odsłuchu:
  1. 01_Human_M_Actor11   → 6 emocji Actor 11 (M)
  2. 02_Human_F_Actor16   → 6 emocji Actor 16 (F)
  3. 03_AI_M_Ryan_Natural → 6 emocji Ryan (M)
  4. 04_AI_F_Sarah_Natural→ 6 emocji Sarah (F)

Jeśli emocja brzmi zbyt podobnie do neutral →


In [5]:
# ============================================================
# SURVEY AUDIO — 24 pliki, pełne pokrycie
# ============================================================

import numpy as np, soundfile as sf, librosa, re, pandas as pd
from pathlib import Path

RAVDESS_DIR    = Path(r"D:\Bachelor Project Code\Audio_Speech_Actors_01-24")
ELEVENLABS_DIR = Path(r"D:\Bachelor Project Code\ElevenLabs_Speech_Voices")
SURVEY_DIR     = Path(r"D:\Bachelor Project Code\Survey_Audio_v2")
CODE_DIR       = Path(r"D:\Bachelor Project Code\Code")
SURVEY_DIR.mkdir(exist_ok=True)

TARGET_RMS = 0.05
EMO_TO_CODE = {"neutral":"01","happy":"03","sad":"04",
               "angry":"05","fearful":"06","disgust":"07"}

def norm_save(src, dst):
    y, sr = librosa.load(str(src), sr=None, mono=True)
    rms = np.sqrt(np.mean(y**2))
    if rms > 0:
        y = y * (TARGET_RMS / rms)
    sf.write(str(dst), np.clip(y,-1,1), sr)
    return rms

def get_ravdess(actor, emotion, statement, rep="01"):
    code  = EMO_TO_CODE[emotion]
    stmt  = "01" if statement=="kids" else "02"
    fname = f"03-01-{code}-01-{stmt}-{rep}-{actor}.wav"
    return RAVDESS_DIR / f"Actor_{actor}" / fname

def get_elevenlabs(voice, emotion, statement, gen="g1"):
    candidates = list(ELEVENLABS_DIR.rglob(
        f"ELv3_{voice}_N_{emotion}_{statement}_{gen}.wav"))
    return candidates[0] if candidates else None

# ── PLAN PLIKÓW ───────────────────────────────────────────────
# Format: (section, source, voice, gender, emotion, statement, rep/gen)
# kids/dogs naprzemiennie per sekcja — 3kids + 3dogs per sekcja

plan = [
    # SEKCJA 1 — Human (Actor11M: neutral/sad/fearful, Actor16F: happy/angry/disgust)
    (1,"human","11","M","neutral", "kids","01"),
    (1,"human","16","F","happy",   "dogs","01"),
    (1,"human","11","M","sad",     "dogs","01"),
    (1,"human","16","F","angry",   "kids","01"),
    (1,"human","11","M","fearful", "kids","01"),
    (1,"human","16","F","disgust", "dogs","01"),

    # SEKCJA 2 — AI (RyanM: neutral/sad/fearful, SarahF: happy/angry/disgust)
    (2,"ai","ryan01",  "M","neutral", "kids","g1"),
    (2,"ai","sarah01", "F","happy",   "dogs","g1"),
    (2,"ai","ryan01",  "M","sad",     "dogs","g1"),
    (2,"ai","sarah01", "F","angry",   "kids","g1"),
    (2,"ai","ryan01",  "M","fearful", "kids","g1"),
    (2,"ai","sarah01", "F","disgust", "dogs","g1"),

    # SEKCJA 3 — Mix: uzupełnienie brakujących emocji
    # Human Actor11M: happy/angry/disgust (inne zdania niż S1)
    (3,"human","11","M","happy",   "dogs","02"),
    (3,"human","11","M","angry",   "kids","02"),
    (3,"human","11","M","disgust", "dogs","02"),
    # Human Actor16F: neutral/sad/fearful (inne zdania niż S1)
    (3,"human","16","F","neutral", "kids","02"),
    (3,"human","16","F","sad",     "dogs","02"),
    (3,"human","16","F","fearful", "kids","02"),
    # AI RyanM: happy/angry/disgust
    (3,"ai","ryan01",  "M","happy",   "dogs","g2"),
    (3,"ai","ryan01",  "M","angry",   "kids","g2"),
    (3,"ai","ryan01",  "M","disgust", "dogs","g2"),
    # AI SarahF: neutral/sad/fearful
    (3,"ai","sarah01", "F","neutral", "kids","g2"),
    (3,"ai","sarah01", "F","sad",     "dogs","g2"),
    (3,"ai","sarah01", "F","fearful", "kids","g2"),
]

# ── GENERUJ PLIKI ─────────────────────────────────────────────
records = []
print(f"Generating {len(plan)} files...\n")

for section, source, voice, gender, emotion, statement, rep in plan:

    # Nazwa pliku: S{n}_{source}_{gender}_{emotion}_{statement}.wav
    src_label = "H" if source=="human" else "AI"
    fname = f"S{section}_{src_label}_{gender}_{emotion}_{statement}.wav"
    dst   = SURVEY_DIR / fname

    if source == "human":
        src = get_ravdess(voice, emotion, statement, rep)
    else:
        src = get_elevenlabs(voice, emotion, statement, rep)

    if src and Path(src).exists():
        orig_rms = norm_save(src, dst)
        status = "✓"
    else:
        orig_rms = 0
        status = "✗ MISSING"

    records.append({
        "section": section, "source": source,
        "voice": voice, "gender": gender,
        "emotion": emotion, "statement": statement,
        "survey_filename": fname,
        "original_path": str(src) if src else "MISSING",
        "status": status
    })
    print(f"  {status} {fname}")

# ── SPRAWDŹ POKRYCIE ──────────────────────────────────────────
df = pd.DataFrame(records)
df.to_csv(CODE_DIR / "survey_files_v2.csv", index=False)

print(f"\n{'='*55}")
print(f"TOTAL: {len(df)} files")
print(f"\nBalance:")
print(f"  Source:    {df.groupby('source').size().to_dict()}")
print(f"  Gender:    {df.groupby('gender').size().to_dict()}")
print(f"  Statement: {df.groupby('statement').size().to_dict()}")
print(f"  Section:   {df.groupby('section').size().to_dict()}")

print(f"\nEmotion coverage per voice:")
for voice in ["11","16","ryan01","sarah01"]:
    emotions = df[df["voice"]==voice]["emotion"].tolist()
    print(f"  {voice:10s}: {sorted(emotions)}")

missing = df[df["status"].str.contains("MISSING")]
if len(missing) > 0:
    print(f"\nMISSING ({len(missing)}):")
    print(missing[["voice","emotion","statement"]].to_string())
else:
    print(f"\nAll files OK. Saved to: {SURVEY_DIR}")

Generating 24 files...

  ✓ S1_H_M_neutral_kids.wav
  ✓ S1_H_F_happy_dogs.wav
  ✓ S1_H_M_sad_dogs.wav
  ✓ S1_H_F_angry_kids.wav
  ✓ S1_H_M_fearful_kids.wav
  ✓ S1_H_F_disgust_dogs.wav
  ✓ S2_AI_M_neutral_kids.wav
  ✓ S2_AI_F_happy_dogs.wav
  ✓ S2_AI_M_sad_dogs.wav
  ✓ S2_AI_F_angry_kids.wav
  ✓ S2_AI_M_fearful_kids.wav
  ✓ S2_AI_F_disgust_dogs.wav
  ✓ S3_H_M_happy_dogs.wav
  ✓ S3_H_M_angry_kids.wav
  ✓ S3_H_M_disgust_dogs.wav
  ✓ S3_H_F_neutral_kids.wav
  ✓ S3_H_F_sad_dogs.wav
  ✓ S3_H_F_fearful_kids.wav
  ✓ S3_AI_M_happy_dogs.wav
  ✓ S3_AI_M_angry_kids.wav
  ✓ S3_AI_M_disgust_dogs.wav
  ✓ S3_AI_F_neutral_kids.wav
  ✓ S3_AI_F_sad_dogs.wav
  ✓ S3_AI_F_fearful_kids.wav

TOTAL: 24 files

Balance:
  Source:    {'ai': 12, 'human': 12}
  Gender:    {'F': 12, 'M': 12}
  Statement: {'dogs': 12, 'kids': 12}
  Section:   {1: 6, 2: 6, 3: 12}

Emotion coverage per voice:
  11        : ['angry', 'disgust', 'fearful', 'happy', 'neutral', 'sad']
  16        : ['angry', 'disgust', 'fearful', 'happy', 

In [ ]:
# ============================================================
# GOOGLE DRIVE — instrukcja i lista plików do wgrania
# ============================================================

import pandas as pd
from pathlib import Path

CODE_DIR   = Path(r"D:\Bachelor Project Code\Code")
SURVEY_DIR = Path(r"D:\Bachelor Project Code\Survey_Audio_v2")

df = pd.read_csv(CODE_DIR / "survey_files_v2.csv")
df_sorted = df.sort_values(["section","source","gender","emotion"])

print("FILES TO UPLOAD TO GOOGLE DRIVE")
print(f"Folder: Survey_Audio_v2\n")
print(f"{'#':>3}  {'Filename':<45} {'Section'} {'Source'} {'Emotion'}")
print("-" * 75)

for i, row in df_sorted.iterrows():
    print(f"{i+1:>3}.  {row['survey_filename']:<45} "
          f"S{row['section']}      {row['source']:<7} {row['emotion']}")

print(f"\nTotal: {len(df)} files")
print(f"\nAfter uploading to Drive:")
print(f"  1. Select all files")
print(f"  2. Right click → Share → Anyone with link → Viewer")
print(f"  3. For each file: right click → Copy link")
print(f"  4. Paste links below in format:")
print(f"     filename.wav → https://drive.google.com/file/d/XXXXX/view")

FILES TO UPLOAD TO GOOGLE DRIVE
Folder: Survey_Audio_v2

  #  Filename                                      Section Source Emotion
---------------------------------------------------------------------------
  4.  S1_H_F_angry_kids.wav                         S1      human   angry
  6.  S1_H_F_disgust_dogs.wav                       S1      human   disgust
  2.  S1_H_F_happy_dogs.wav                         S1      human   happy
  5.  S1_H_M_fearful_kids.wav                       S1      human   fearful
  1.  S1_H_M_neutral_kids.wav                       S1      human   neutral
  3.  S1_H_M_sad_dogs.wav                           S1      human   sad
 10.  S2_AI_F_angry_kids.wav                        S2      ai      angry
 12.  S2_AI_F_disgust_dogs.wav                      S2      ai      disgust
  8.  S2_AI_F_happy_dogs.wav                        S2      ai      happy
 11.  S2_AI_M_fearful_kids.wav                      S2      ai      fearful
  7.  S2_AI_M_neutral_kids.wav               

: 